# RAPIRO — Entrenamiento CNN desde Google Drive

Entrena MobileNetV2 (5 clases) leyendo las fotos directamente desde la carpeta de Drive del equipo.

**Clases:**
| ID | Carpeta en Drive | Descripción |
|----|-----------------|-------------|
| 0  | Estudiando      | Concentrado, trabajando |
| 1  | Usando el Cel   | Uso de celular |
| 2  | Puesto vacio    | Ausente |
| 3  | Confundido      | Ceño fruncido, mano en barbilla |
| 4  | Aburrido        | Postura caída, mirada perdida |

**Pasos:** Ejecutar todas las celdas en orden (`Runtime > Run all`).

In [ ]:
# ── 1. Montar Drive y autenticar ──────────────────────────────────────────────
from google.colab import drive, auth
drive.mount('/content/drive')
auth.authenticate_user()
print('Drive montado y usuario autenticado.')

In [ ]:
# ── 2. Instalar dependencias ──────────────────────────────────────────────────
!pip install -q tensorflow matplotlib

In [ ]:
# ── 3. Descargar dataset desde Drive (por folder ID) ─────────────────────────
import os, io, shutil
from googleapiclient.discovery import build
from googleapiclient.http import MediaIoBaseDownload

# ID de la carpeta raíz del dataset en Drive
DRIVE_FOLDER_ID = '1ULuPEGf_U_qH5OwrI1ps8kk1pHdq8mrP'

# Mapeo nombre carpeta Drive → class_X (fuerza el orden correcto de clases)
FOLDER_MAP = {
    'Estudiando':    'class_0',
    'Usando el Cel': 'class_1',
    'Confundido':    'class_3',
    'Aburrido':      'class_4',
}
# Puesto vacio puede tener espacios extra — lo matcheamos por strip()
PUESTO_VACIO_KEY = 'Puesto vacio'

LOCAL_DATASET = '/content/dataset'
service = build('drive', 'v3', cache_discovery=False)


def list_folder(folder_id):
    items, token = [], None
    while True:
        resp = service.files().list(
            q=f"'{folder_id}' in parents and trashed=false",
            fields='nextPageToken, files(id, name, mimeType)',
            pageToken=token,
        ).execute()
        items += resp.get('files', [])
        token = resp.get('nextPageToken')
        if not token:
            break
    return items


def download_file(file_id, dest_path):
    request = service.files().get_media(fileId=file_id)
    with io.FileIO(dest_path, 'wb') as fh:
        downloader = MediaIoBaseDownload(fh, request, chunksize=4*1024*1024)
        done = False
        while not done:
            _, done = downloader.next_chunk()


# Limpiar y recrear carpeta local
if os.path.exists(LOCAL_DATASET):
    shutil.rmtree(LOCAL_DATASET)

# Listar subcarpetas del dataset en Drive
subfolders = list_folder(DRIVE_FOLDER_ID)
total_images = 0

for folder in subfolders:
    if folder['mimeType'] != 'application/vnd.google-apps.folder':
        continue

    name = folder['name'].strip()

    # Resolver nombre a class_X
    if name in FOLDER_MAP:
        class_dir = FOLDER_MAP[name]
    elif name.startswith(PUESTO_VACIO_KEY):
        class_dir = 'class_2'
    else:
        print(f'  [!] Carpeta desconocida ignorada: {name}')
        continue

    dest = os.path.join(LOCAL_DATASET, class_dir)
    os.makedirs(dest, exist_ok=True)

    images = list_folder(folder['id'])
    count = 0
    for img in images:
        if img['mimeType'].startswith('image/'):
            download_file(img['id'], os.path.join(dest, img['name']))
            count += 1
    total_images += count
    print(f'  {class_dir} ({name}): {count} imágenes descargadas')

print(f'\nTotal: {total_images} imágenes en {LOCAL_DATASET}')

In [ ]:
# ── 4. Configuración ──────────────────────────────────────────────────────────
import numpy as np
import tensorflow as tf

IMG_SIZE    = (224, 224)
NUM_CLASSES = 5
CLASS_NAMES = ['Estudiando', 'Usando celular', 'Puesto vacio', 'Confundido', 'Aburrido']
BATCH_SIZE  = 32
EPOCHS_P1   = 15   # fase 1: solo cabeza
EPOCHS_P2   = 8    # fase 2: fine-tuning (opcional)
DO_FINETUNE = True

MODELS_DIR  = '/content/drive/MyDrive/RAPIRO_models'
os.makedirs(MODELS_DIR, exist_ok=True)

print(f'TensorFlow {tf.__version__}')
print(f'GPU disponible: {len(tf.config.list_physical_devices("GPU")) > 0}')

In [ ]:
# ── 5. Cargar dataset con augmentación ───────────────────────────────────────
train_datagen = tf.keras.preprocessing.image.ImageDataGenerator(
    rescale=1.0/255,
    validation_split=0.2,
    rotation_range=15,
    width_shift_range=0.1,
    height_shift_range=0.1,
    zoom_range=0.15,
    horizontal_flip=True,
    brightness_range=[0.7, 1.3],
    shear_range=0.1,
)
val_datagen = tf.keras.preprocessing.image.ImageDataGenerator(
    rescale=1.0/255,
    validation_split=0.2,
)

# classes=['class_0',...] fuerza el orden correcto independiente de nombres en disco
CLASSES_ORDER = ['class_0', 'class_1', 'class_2', 'class_3', 'class_4']

train_gen = train_datagen.flow_from_directory(
    LOCAL_DATASET,
    target_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    class_mode='categorical',
    classes=CLASSES_ORDER,
    subset='training',
    shuffle=True,
    seed=42,
)
val_gen = val_datagen.flow_from_directory(
    LOCAL_DATASET,
    target_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    class_mode='categorical',
    classes=CLASSES_ORDER,
    subset='validation',
    shuffle=False,
    seed=42,
)

print(f'Train: {train_gen.samples} imgs | Val: {val_gen.samples} imgs')
print(f'Clases: {train_gen.class_indices}')

In [ ]:
# ── 6. Construir modelo ───────────────────────────────────────────────────────
def build_model(fine_tune_layers=0):
    base = tf.keras.applications.MobileNetV2(
        input_shape=(*IMG_SIZE, 3),
        include_top=False,
        weights='imagenet',
    )
    base.trainable = False
    if fine_tune_layers > 0:
        for layer in base.layers[-fine_tune_layers:]:
            layer.trainable = True

    inputs  = tf.keras.Input(shape=(*IMG_SIZE, 3))
    x       = base(inputs, training=False)
    x       = tf.keras.layers.GlobalAveragePooling2D()(x)
    x       = tf.keras.layers.Dense(256, activation='relu')(x)
    x       = tf.keras.layers.Dropout(0.4)(x)
    outputs = tf.keras.layers.Dense(NUM_CLASSES, activation='softmax')(x)
    return tf.keras.Model(inputs, outputs)


callbacks = [
    tf.keras.callbacks.ModelCheckpoint(
        '/content/best_model.h5', save_best_only=True,
        monitor='val_accuracy', verbose=1,
    ),
    tf.keras.callbacks.EarlyStopping(
        monitor='val_accuracy', patience=5, restore_best_weights=True, verbose=1,
    ),
    tf.keras.callbacks.ReduceLROnPlateau(
        monitor='val_loss', factor=0.5, patience=3, verbose=1,
    ),
]

model = build_model()
model.summary()

In [ ]:
# ── 7. Fase 1: entrenar solo la cabeza ───────────────────────────────────────
model.compile(
    optimizer=tf.keras.optimizers.Adam(1e-3),
    loss='categorical_crossentropy',
    metrics=['accuracy'],
)
history1 = model.fit(
    train_gen, validation_data=val_gen,
    epochs=EPOCHS_P1, callbacks=callbacks,
)
loss1, acc1 = model.evaluate(val_gen, verbose=0)
print(f'\nFase 1 — Val accuracy: {acc1*100:.2f}%')

In [ ]:
# ── 8. Fase 2: fine-tuning (opcional) ────────────────────────────────────────
history2 = None
if DO_FINETUNE:
    print('Fase 2: descongelando ultimas 30 capas del base...')
    model = build_model(fine_tune_layers=30)
    model.compile(
        optimizer=tf.keras.optimizers.Adam(1e-5),
        loss='categorical_crossentropy',
        metrics=['accuracy'],
    )
    history2 = model.fit(
        train_gen, validation_data=val_gen,
        epochs=EPOCHS_P2, callbacks=callbacks,
    )
    loss2, acc2 = model.evaluate(val_gen, verbose=0)
    print(f'Fase 2 — Val accuracy: {acc2*100:.2f}%')

In [ ]:
# ── 9. Convertir a TFLite INT8 ───────────────────────────────────────────────
LOCAL_TFLITE = '/content/mobilenetv2_int8.tflite'

converter = tf.lite.TFLiteConverter.from_keras_model(model)
converter.optimizations = [tf.lite.Optimize.DEFAULT]

def representative_data_gen():
    count = 0
    for batch_images, _ in train_gen:
        for img in batch_images:
            yield [np.expand_dims(img.astype(np.float32), axis=0)]
            count += 1
            if count >= 200:
                return

converter.representative_dataset = representative_data_gen
converter.target_spec.supported_ops = [tf.lite.OpsSet.TFLITE_BUILTINS_INT8]
converter.inference_input_type  = tf.int8
converter.inference_output_type = tf.int8

tflite_model = converter.convert()
with open(LOCAL_TFLITE, 'wb') as f:
    f.write(tflite_model)

size_kb = os.path.getsize(LOCAL_TFLITE) / 1024
print(f'Modelo INT8: {size_kb:.1f} KB')

In [ ]:
# ── 10. Guardar modelo en Drive ───────────────────────────────────────────────
import shutil
drive_tflite = os.path.join(MODELS_DIR, 'mobilenetv2_int8.tflite')
shutil.copy(LOCAL_TFLITE, drive_tflite)
print(f'Modelo guardado en Drive: {drive_tflite}')
print()
print('Para copiarlo a la Raspberry Pi:')
print('  scp <ruta_local_descargada> pi@<IP_RAPIRO>:~/TPI-RAPIRO-UCP/models/mobilenetv2_int8.tflite')

In [ ]:
# ── 11. Curvas de entrenamiento ───────────────────────────────────────────────
import matplotlib.pyplot as plt

acc  = history1.history['accuracy']
val  = history1.history['val_accuracy']
loss = history1.history['loss']
vloss= history1.history['val_loss']

if history2:
    sep = len(acc)
    acc  += history2.history['accuracy']
    val  += history2.history['val_accuracy']
    loss += history2.history['loss']
    vloss+= history2.history['val_loss']

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

if history2:
    axes[0].axvline(sep, color='gray', linestyle='--', label='fine-tune')
    axes[1].axvline(sep, color='gray', linestyle='--')

axes[0].plot(acc, label='train'); axes[0].plot(val, label='val')
axes[0].set_title('Accuracy'); axes[0].legend(); axes[0].grid(True)

axes[1].plot(loss, label='train'); axes[1].plot(vloss, label='val')
axes[1].set_title('Loss'); axes[1].legend(); axes[1].grid(True)

plt.tight_layout()
plot_path = os.path.join(MODELS_DIR, 'training_curves.png')
plt.savefig(plot_path, dpi=120)
plt.show()
print(f'Curvas guardadas en Drive: {plot_path}')